# brain_spi — Colab quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/neuroneural/brain-spi/blob/main/examples/colab_quickstart.ipynb)

Run the `brain_spi` pipeline on public fMRI ICA data, entirely in Colab — nothing
to clone. **ABIDE** (controls vs. autism) by default; set `DATASET = 'cobre'`
(schizophrenia) to switch.

* **Part 1** — compute SPIs and look at their mean connectivity
* **Part 2** — the significant-differences pipeline (full cohort)

The first cell installs `brain_spi` (and `pyspi`); give it a few minutes.

In [ ]:
# Install the package and grab the dataset helper (one-time, ~a few minutes).
!pip install -q "git+https://github.com/neuroneural/brain-spi.git"
!wget -q -O datasets.py https://raw.githubusercontent.com/neuroneural/brain-spi/main/examples/datasets.py

In [ ]:
import logging
import numpy as np
import matplotlib.pyplot as plt
logging.basicConfig(level=logging.INFO, format='%(message)s')  # per-SPI progress

from brain_spi import BrainSPI
from brain_spi.viz import plot_heatmap
import datasets   # downloaded above: download() + load() + domain_spec()

DATASET = 'abide'   # or 'cobre'
FAST_SPIS = ['cov_GraphicalLassoCV', 'spearmanr', 'kendalltau', 'xcorr_mean_sig-True', 'pec']
print('ready  |  dataset:', DATASET, '|  SPIs:', FAST_SPIS)

In [ ]:
def balanced_subsample(data, labels, per_group=40, seed=0):
    """Equal number of subjects per label (for a fast demo)."""
    rng = np.random.default_rng(seed)
    idx = np.concatenate([
        rng.choice(np.where(labels == c)[0], size=per_group, replace=False)
        for c in np.unique(labels)
    ])
    return data[idx], labels[idx]

datasets.download(DATASET)                   # downloads to the Colab VM (cached for the session)
data, labels, group_names = datasets.load(DATASET)
domain_spec = datasets.domain_spec(DATASET)  # 53 ICN components -> network bands
print(f'{DATASET}: data {data.shape}, labels {np.bincount(labels)}, groups {group_names}')

## Part 1 — Compute SPIs and look at them

`BrainSPI().fit()` computes one connectivity matrix per SPI per subject. Here we
inspect the connectivity itself — per-group mean matrices — on a small balanced
subsample to keep it quick.

In [ ]:
d, y = balanced_subsample(data, labels, per_group=40)

pipe = BrainSPI(spis=FAST_SPIS, group_names=group_names)
result = pipe.fit(d, y)

result   # rich repr — SPIs, shapes, and .help()

In [ ]:
# Mean connectivity for one SPI, split by group
result['kendalltau'].plot_mean(by_group=True, group_names=group_names, domain_spec=domain_spec)
plt.show()

In [ ]:
# Mean connectivity across all SPIs, side by side
spis = result.spis
fig, axes = plt.subplots(1, len(spis), figsize=(3.2 * len(spis), 3.4))
for ax, name in zip(axes, spis):
    plot_heatmap(result[name].mean_matrix(), ax=ax, guides=domain_spec, title=name)
fig.suptitle(f'{DATASET.upper()} — mean connectivity per SPI', y=1.04)
fig.tight_layout(); plt.show()

## Part 2 — Significant-differences pipeline (full cohort)

Per SPI: Welch t-test (Bonferroni → `p_thresh`) AND a Random-Forest importance
mask (`rf_mask`), intersected (`and_mask`). Averaging the AND masks across SPIs
gives the cross-SPI **aggregate**. Runs on all subjects (first fit takes a few
minutes; cached afterward).

In [ ]:
pipe_full = BrainSPI(spis=FAST_SPIS, group_names=group_names)
result_full = pipe_full.fit(data, labels)   # full cohort

result_full['kendalltau'].plot_triptych(domain_spec=domain_spec)   # sig-p | RF | AND
plt.show()

In [ ]:
# Cross-SPI aggregate (lazy)
agg = result_full.aggregate
print('edges flagged by >=1 SPI:', int((agg.mean_and > 0).sum() // 2))
agg.plot(domain_spec=domain_spec)
plt.show()

In [ ]:
# Robustness: average of 20 subject-subset cross-SPI AND maps (same inferno scale)
boot = result_full.bootstrap(n=20, frac=0.66)
fig, ax = plt.subplots(figsize=(6, 5))
im = plot_heatmap(boot.mean, ax=ax, cmap='inferno', vmin=0, vmax=float(boot.mean.max()),
                  guides=domain_spec, title=f'{DATASET.upper()} — mean AND over 20 subject subsets')
fig.colorbar(im, ax=ax, fraction=0.045); plt.show()

# Save the result (portable .npz). On Colab, download it to keep it past the session:
result_full.to_npz(f'{DATASET}_result.npz')
# from google.colab import files; files.download(f'{DATASET}_result.npz')
print(f'saved -> {DATASET}_result.npz')